# 02 - Retrieval
BM25 vs dense vs hybrid (reciprocal rank fusion). The first run embeds ~5k abstracts and caches them in `artifacts/index`.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
from src import data, evaluate
from src.retrieval import Retriever

r = Retriever()
val = data.claims('validation')

In [ ]:
res = pd.DataFrame({m: evaluate.retrieval(r, val, mode=m) for m in ('bm25', 'dense', 'hybrid')}).T
res.round(3)

## Where does hybrid still miss?
These are the claims whose gold abstract isn't in the top 5. Look for patterns (abbreviations, indirect wording) before reaching for a bigger model.

In [ ]:
gold = val[val.gold_docs.map(len) > 0]
miss = [(c, g) for c, g in zip(gold.claim, gold.gold_docs) if not set(g) & set(r.search(c, 5).doc_id.tolist())]
print(len(miss), 'of', len(gold))
for c, g in miss[:10]:
    print('-', c, '| gold docs', g)